Phase 2

collect files and 70/15/15 split



In [ ]:
# Phase 2 — Data Pre-processing and Feature Extraction

from collections import Counter
from sklearn.model_selection import train_test_split
composers = ['bach', 'beethoven', 'chopin', 'mozart']
composer_labels = {name: i for i, name in enumerate(composers)}
files = []

# Combine the existing folders before making a new 70/15/15 split.
for split in ['train', 'dev', 'test']:
    for composer in composers:
        folder = os.path.join(DATA_DIR, split, composer)

        if not os.path.isdir(folder):
            continue

        for file in os.listdir(folder):
            if file.lower().endswith(('.mid', '.midi')):
                files.append([
                    composer,
                    composer_labels[composer],
                    os.path.join(folder, file)
                ])

files_df = pd.DataFrame(
    files,
    columns=['composer', 'label', 'path']
)

missing = [
    name for name in composers
    if name not in files_df['composer'].unique()
]

if missing:
    raise ValueError(f"Missing composers: {missing}")

# Stratify keeps the composer distribution balanced in each split.
train_files, remaining = train_test_split(
    files_df,
    test_size=0.30,
    random_state=42,
    stratify=files_df['composer']
)

val_files, test_files = train_test_split(
    remaining,
    test_size=0.50,
    random_state=42,
    stratify=remaining['composer']
)

print("Train:", len(train_files))
print("Validation:", len(val_files))
print("Test:", len(test_files))

In [ ]:


def read_song(path):
    midi = pretty_midi.PrettyMIDI(path)

    notes = []
    chord_times = {}

    for instrument in midi.instruments:
        if instrument.is_drum:
            continue

        for note in instrument.notes:
            duration = note.end - note.start
            notes.append((note.start, note.pitch, duration))

            # Notes starting together are treated as a chord.
            time = round(note.start, 3)
            chord_times.setdefault(time, []).append(note.pitch)

    notes.sort()

    pitches = [pitch for start, pitch, duration in notes]
    durations = [duration for start, pitch, duration in notes]
    offsets = [start for start, pitch, duration in notes]
    tokens = [f"P{pitch}" for pitch in pitches]

    chords = [
        '.'.join(map(str, sorted(set(pitches))))
        for pitches in chord_times.values()
        if len(set(pitches)) > 1
    ]

    tempo_values = midi.get_tempo_changes()[1]
    tempo = np.mean(tempo_values) if len(tempo_values) else 120

    return {
        'tokens': tokens,
        'pitches': pitches,
        'durations': durations,
        'offsets': offsets,
        'chords': chords,
        'tempo': tempo
    }


# Save extracted features so each MIDI file is read only once.
song_data = {}

for path in files_df['path']:
    try:
        song_data[path] = read_song(path)
    except Exception as error:
        print("Skipped:", path, error)

sample = song_data[train_files.iloc[0]['path']]

print("Sample notes:", len(sample['tokens']))
print("Sample chords:", len(sample['chords']))
print("Sample tempo:", round(sample['tempo'], 2))


Extract notes, chords, and tempo

In [ ]:
# Extract MIDI Features

def read_song(path):
    midi = pretty_midi.PrettyMIDI(path)

    notes = []
    chord_times = {}

    for instrument in midi.instruments:
        if instrument.is_drum:
            continue

        for note in instrument.notes:
            duration = note.end - note.start
            notes.append((note.start, note.pitch, duration))

            # Notes starting together are treated as a chord.
            time = round(note.start, 3)
            chord_times.setdefault(time, []).append(note.pitch)

    notes.sort()

    pitches = [pitch for start, pitch, duration in notes]
    durations = [duration for start, pitch, duration in notes]
    offsets = [start for start, pitch, duration in notes]
    tokens = [f"P{pitch}" for pitch in pitches]

    chords = [
        '.'.join(map(str, sorted(set(pitches))))
        for pitches in chord_times.values()
        if len(set(pitches)) > 1
    ]

    tempo_values = midi.get_tempo_changes()[1]
    tempo = np.mean(tempo_values) if len(tempo_values) else 120

    return {
        'tokens': tokens,
        'pitches': pitches,
        'durations': durations,
        'offsets': offsets,
        'chords': chords,
        'tempo': tempo
    }


# Save extracted features so each MIDI file is read only once.
song_data = {}

for path in files_df['path']:
    try:
        song_data[path] = read_song(path)
    except Exception as error:
        print("Skipped:", path, error)

sample = song_data[train_files.iloc[0]['path']]

print("Sample notes:", len(sample['tokens']))
print("Sample chords:", len(sample['chords']))
print("Sample tempo:", round(sample['tempo'], 2))

In [ ]:
#  NLP Note Encoding

def make_ngrams(tokens, n):
    return [
        '_'.join(tokens[i:i + n])
        for i in range(len(tokens) - n + 1)
    ]
unigrams = Counter()
bigrams = Counter()
trigrams = Counter()

# Build the vocabulary using training data only.
for path in train_files['path']:
    tokens = song_data[path]['tokens']
    unigrams.update(tokens)
    bigrams.update(make_ngrams(tokens, 2))
    trigrams.update(make_ngrams(tokens, 3))

note_to_number = {
    note: number + 1
    for number, note in enumerate(sorted(unigrams))
}
print("Unigrams:", len(unigrams))
print("Bigrams:", len(bigrams))
print("Trigrams:", len(trigrams))
print("Common bigrams:", bigrams.most_common(5))
print("Common trigrams:", trigrams.most_common(5))

In [ ]:
# Create 100-Note LSTM Sequences

sequence_length = 100

def create_sequences(file_table):
    X = []
    y = []

    for _, song in file_table.iterrows():
        tokens = song_data[song['path']]['tokens']
        # Convert note tokens such as P60 into integers
        encoded = [
            note_to_number.get(token, 0)
            for token in tokens
        ]
        # Each sequence contains 100 notes and overlaps by 50 notes
        for start in range(0, len(encoded) - 99, 50):
            X.append(encoded[start:start + 100])
            y.append(song['label'])

    return np.array(X), np.array(y)
X_train_lstm, y_train_lstm = create_sequences(train_files)
X_val_lstm, y_val_lstm = create_sequences(val_files)
X_test_lstm, y_test_lstm = create_sequences(test_files)

print("LSTM train:", X_train_lstm.shape)
print("LSTM validation:", X_val_lstm.shape)
print("LSTM test:", X_test_lstm.shape)

In [ ]:
# Create CNN Piano Rolls

def create_roll(path, pitch_shift=0, tempo_scale=1.0):
    midi = pretty_midi.PrettyMIDI(path)
    roll = midi.get_piano_roll(fs=10).astype('float32')

    # Move every pitch up or down for augmentation.
    if pitch_shift:
        roll = np.roll(roll, pitch_shift, axis=0)
        if pitch_shift > 0:
            roll[:pitch_shift] = 0
        else:
            roll[pitch_shift:] = 0
    # Change the number of time frames to simulate tempo changes
    if tempo_scale != 1 and roll.shape[1] > 0:
        new_size = max(1, int(roll.shape[1] / tempo_scale))

        positions = np.linspace(
            0,
            roll.shape[1] - 1,
            new_size
        ).astype(int)

        roll = roll[:, positions]
    # Every  cnn image must have the same 128 × 300 shape
    fixed_roll = np.zeros((128, 300), dtype='float32')
    length = min(300, roll.shape[1])
    fixed_roll[:, :length] = roll[:, :length]

    # Normalize velocity values to the range 0–1
    return np.clip(fixed_roll / 127, 0, 1)

def create_cnn_data(file_table, augment=False):
    X = []
    y = []
    for _, song in file_table.iterrows():
        versions = [(0, 1.0)]
        if augment:
            versions += [(2, 1.0), (0, 1.1)]
        for pitch_shift, tempo_scale in versions:
            X.append(
                create_roll(
                    song['path'],
                    pitch_shift,
                    tempo_scale
                )
            )
            y.append(song['label'])
    # Add one channel because the piano roll is a grayscale image
    return np.array(X)[..., np.newaxis], np.array(y)

X_train_cnn, y_train_cnn = create_cnn_data(
    train_files,
    augment=True
)
X_val_cnn, y_val_cnn = create_cnn_data(val_files)
X_test_cnn, y_test_cnn = create_cnn_data(test_files)

print("CNN train:", X_train_cnn.shape)
print("CNN validation:", X_val_cnn.shape)
print("CNN test:", X_test_cnn.shape)